In [23]:
!pip install --upgrade torch transformers peft


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

In [25]:
model_name="distilbert-base-uncased"
model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)      

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1815.66it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [27]:
print(f"Trainable parameters before LoRA(fully fine-tuned): {count_trainable_parameters(model)}")

Trainable parameters before LoRA(fully fine-tuned): 66955010


In [28]:
for name, param in model.named_parameters():
    print(f"{name};{param.numel()}parameters(trainable={param.requires_grad})")
    

distilbert.embeddings.word_embeddings.weight;23440896parameters(trainable=True)
distilbert.embeddings.position_embeddings.weight;393216parameters(trainable=True)
distilbert.embeddings.LayerNorm.weight;768parameters(trainable=True)
distilbert.embeddings.LayerNorm.bias;768parameters(trainable=True)
distilbert.transformer.layer.0.attention.q_lin.weight;589824parameters(trainable=True)
distilbert.transformer.layer.0.attention.q_lin.bias;768parameters(trainable=True)
distilbert.transformer.layer.0.attention.k_lin.weight;589824parameters(trainable=True)
distilbert.transformer.layer.0.attention.k_lin.bias;768parameters(trainable=True)
distilbert.transformer.layer.0.attention.v_lin.weight;589824parameters(trainable=True)
distilbert.transformer.layer.0.attention.v_lin.bias;768parameters(trainable=True)
distilbert.transformer.layer.0.attention.out_lin.weight;589824parameters(trainable=True)
distilbert.transformer.layer.0.attention.out_lin.bias;768parameters(trainable=True)
distilbert.transformer

In [41]:
lora_config = LoraConfig(
    r=2,
    lora_alpha=512,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    task_type="SEQ_CLS"
)

In [42]:
lora_model = get_peft_model(model, lora_config)

In [43]:
print("trainable parameters after LoRA:")
lora_model.print_trainable_parameters()

trainable parameters after LoRA:
trainable params: 628,994 || all params: 67,584,004 || trainable%: 0.9307


In [39]:
for name, param in model.named_parameters():
    print(f"{name};{param.numel()}parameters(trainable={param.requires_grad})")
        
    

distilbert.embeddings.word_embeddings.weight;23440896parameters(trainable=False)
distilbert.embeddings.position_embeddings.weight;393216parameters(trainable=False)
distilbert.embeddings.LayerNorm.weight;768parameters(trainable=False)
distilbert.embeddings.LayerNorm.bias;768parameters(trainable=False)
distilbert.transformer.layer.0.attention.q_lin.base_layer.weight;589824parameters(trainable=False)
distilbert.transformer.layer.0.attention.q_lin.base_layer.bias;768parameters(trainable=False)
distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight;6144parameters(trainable=True)
distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight;6144parameters(trainable=True)
distilbert.transformer.layer.0.attention.k_lin.weight;589824parameters(trainable=False)
distilbert.transformer.layer.0.attention.k_lin.bias;768parameters(trainable=False)
distilbert.transformer.layer.0.attention.v_lin.base_layer.weight;589824parameters(trainable=False)
distilbert.transformer.layer.0.atte